In [1]:
import pdfplumber
import pandas as pd
import re

# ------------------------
# Configuración y patrones
# ------------------------
pdf_path = "Remittance_Cenco.pdf"

# Vouchers reconocidos
VOUCHERS = r"(FS|CH|DEC|NC|ND|LTG|FPM|DCA|DCF|DND|DAV|DCC|RPL|DAT|DEV|DPC)"
pat_inicio = re.compile(rf"^{VOUCHERS}\b", flags=re.IGNORECASE)

# Patrones para documentos específicos
pat_documento_dec = re.compile(r"\b(\d{4}-\d{7,20})\b")     # DEC
pat_documento_pmp = re.compile(r"\b(PMP\d{3,20})\b")        # PMP / LTG / FPM
pat_documento_fs_pair = re.compile(r"\b([A-Z0-9]{2,6})\s+(\d{5,12})\b")  # FS
pat_fecha = re.compile(r"\d{2}/\d{2}/\d{4}")                # Fecha

# Secciones posibles
SECCIONES = {"PERFUME", "DROGUE", "RANCHO", "PLATOS"}

# -----------------------
# Funciones utilitarias
# -----------------------

def normalize_whitespace(s: str) -> str:
    """Quita espacios extra y normaliza la cadena."""
    return re.sub(r"\s+", " ", s).strip()

def is_start_of_record(line: str) -> bool:
    """Verifica si una línea comienza con un voucher reconocido."""
    return bool(pat_inicio.match(line.strip()))

def merge_lines(lines):
    """
    Une todas las líneas de un mismo registro:
    - Si empieza con voucher → nuevo registro
    - Si no, se agrega al registro actual
    Esto asegura que multi-líneas en DESCRIPCION o TIENDA queden juntas.
    """
    registros = []
    buffer = ""

    for raw in lines:
        line = raw.strip()
        if not line:
            continue

        if pat_inicio.match(line):
            if buffer:
                registros.append(normalize_whitespace(buffer))
            buffer = line
        else:
            buffer += " " + line  # unir toda línea que no empieza con voucher

    if buffer:
        registros.append(normalize_whitespace(buffer))
    return registros

def extract_numeric_values_after_date(text: str):
    """Extrae todos los valores numéricos después de la fecha en la línea."""
    t = re.sub(r"[^\d\.,\-]+", " ", text)
    toks = [x for x in t.split() if x.strip() != ""]
    return [tk for tk in toks if re.search(r"\d", tk)]

# ------------------------
# Parsers por tipo de VOUCHER
# ------------------------

def parse_DEC(line, voucher):
    """Parsea registros tipo DEC, incluyendo especiales 'DTO POR ESCALA VOLU'."""
    line = normalize_whitespace(line)
    mfecha = pat_fecha.search(line)
    if not mfecha:
        return None
    fecha = mfecha.group(0)

    pre_date = line[len(voucher):mfecha.start()].strip()
    post_date = line[mfecha.end():].strip()

    prefix = "DTO POR ESCALA VOLU"
    if pre_date.startswith(prefix):
        descripcion = prefix
        rest = pre_date[len(prefix):].strip()
        # Capturar documento DEC seguido de todo lo que queda como tienda
        mdoc = re.match(r"(\d{4}-\d{7,20})(.*)", rest)
        if mdoc:
            documento = mdoc.group(1)
            tienda = normalize_whitespace(mdoc.group(2))
        else:
            documento = ""
            tienda = rest
    else:
        # Caso genérico DEC
        mdoc = pat_documento_dec.search(pre_date)
        if mdoc:
            documento = mdoc.group(1)
            descripcion = normalize_whitespace(pre_date[:mdoc.start()])
            tienda = normalize_whitespace(pre_date[mdoc.end():])
        else:
            documento = ""
            descripcion = pre_date
            tienda = ""

    # Extraer valores numéricos después de la fecha
    nums = extract_numeric_values_after_date(post_date)
    while len(nums) < 8:
        nums.append("0")
    valor, iva, ret_fuente, ret_iva, ret_ica, otros_imp, valor_pag = nums[:7]
    doc_soporte = nums[7]

    # Detectar sección dentro de DESCRIPCION o TIENDA
    seccion = ""
    for sec in SECCIONES:
        if sec in descripcion:
            seccion = sec
            descripcion = descripcion.replace(sec, "").strip()
            break
        elif sec in tienda:
            seccion = sec
            tienda = tienda.replace(sec, "").strip()
            break

    return {
        "VOUCHER": voucher,
        "DESCRIPCION": descripcion,
        "DOCUMENTO": documento,
        "TIENDA": tienda,
        "SECCION": seccion.upper(),
        "F. REGISTRO": fecha,
        "VALOR": valor,
        "IVA": iva,
        "RET. FUENTE": ret_fuente,
        "RET. IVA": ret_iva,
        "RET. ICA": ret_ica,
        "OTROS IMP.": otros_imp,
        "VALOR PAG": valor_pag,
        "DOC.SOPORTE": doc_soporte
    }

def parse_PMP(line, voucher):
    """Parsea vouchers tipo LTG y FPM."""
    m = pat_documento_pmp.search(line)
    if not m:
        return None
    documento = m.group(1)
    doc_start, doc_end = m.start(), m.end()
    descripcion = normalize_whitespace(line[len(voucher):doc_start])

    tail = line[doc_end:].strip()
    tokens_tail = tail.split()
    idx_sec = None
    for i, t in enumerate(tokens_tail):
        if t.upper() in SECCIONES:
            idx_sec = i
            break

    if idx_sec is None:
        mfecha = pat_fecha.search(tail)
        if not mfecha:
            return None
        fecha = mfecha.group(0)
        before_date = tail.split(fecha, 1)[0].strip()
        btoks = before_date.split()
        if btoks and btoks[-1].upper() in SECCIONES:
            seccion = btoks[-1]
            tienda = " ".join(btoks[:-1]).strip()
        else:
            tienda = before_date
            seccion = ""
    else:
        tienda = " ".join(tokens_tail[:idx_sec]).strip()
        seccion = tokens_tail[idx_sec]

    mfecha = pat_fecha.search(tail)
    fecha = mfecha.group(0)

    nums = extract_numeric_values_after_date(tail.split(fecha,1)[1])
    while len(nums) < 8:
        nums.append("0")
    valor, iva, ret_fuente, ret_iva, ret_ica, otros_imp, valor_pag = nums[:7]
    doc_soporte = nums[7]

    return {
        "VOUCHER": voucher,
        "DESCRIPCION": descripcion,
        "DOCUMENTO": documento,
        "TIENDA": tienda,
        "SECCION": seccion,
        "F. REGISTRO": fecha,
        "VALOR": valor,
        "IVA": iva,
        "RET. FUENTE": ret_fuente,
        "RET. IVA": ret_iva,
        "RET. ICA": ret_ica,
        "OTROS IMP.": otros_imp,
        "VALOR PAG": valor_pag,
        "DOC.SOPORTE": doc_soporte
    }

def parse_FS(line, voucher):
    """Parsea vouchers tipo FS."""
    m = pat_documento_fs_pair.search(line)
    if not m:
        return None
    documento = f"{m.group(1)} {m.group(2)}"
    doc_start = m.start()
    descripcion = normalize_whitespace(line[len(voucher):doc_start])

    tail = line[m.end():].strip()
    mfecha = pat_fecha.search(tail)
    if not mfecha:
        return None
    fecha = mfecha.group(0)
    before_date = tail.split(fecha,1)[0].strip()

    adm_idx = re.search(r"\bADM\.", before_date, flags=re.IGNORECASE)
    tienda = before_date[adm_idx.start():].strip() if adm_idx else before_date

    nums = extract_numeric_values_after_date(tail.split(fecha,1)[1])
    while len(nums) < 8:
        nums.append("0")
    valor, iva, ret_fuente, ret_iva, ret_ica, otros_imp, valor_pag = nums[:7]
    doc_soporte = nums[7]

    return {
        "VOUCHER": voucher,
        "DESCRIPCION": descripcion,
        "DOCUMENTO": documento,
        "TIENDA": tienda,
        "SECCION": "",
        "F. REGISTRO": fecha,
        "VALOR": valor,
        "IVA": iva,
        "RET. FUENTE": ret_fuente,
        "RET. IVA": ret_iva,
        "RET. ICA": ret_ica,
        "OTROS IMP.": otros_imp,
        "VALOR PAG": valor_pag,
        "DOC.SOPORTE": doc_soporte
    }

def parse_CH(line, voucher):
    """Parsea vouchers tipo CH o registros con sección fija (RPL, DCA, etc.)."""
    mfecha = pat_fecha.search(line)
    if not mfecha:
        return None
    fecha = mfecha.group(0)
    pre_date = line[len(voucher):mfecha.start()].strip()
    adm_idx = re.search(r"\bADM\.", pre_date, flags=re.IGNORECASE)
    if adm_idx:
        descripcion = normalize_whitespace(pre_date[:adm_idx.start()])
        tienda = pre_date[adm_idx.start():].strip()
    else:
        descripcion = pre_date
        tienda = ""

    # Extraer valores numéricos
    nums = extract_numeric_values_after_date(line[mfecha.end():])
    while len(nums) < 8:
        nums.append("0")
    valor, iva, ret_fuente, ret_iva, ret_ica, otros_imp, valor_pag = nums[:7]
    doc_soporte = nums[7]

    return {
        "VOUCHER": voucher,
        "DESCRIPCION": descripcion,
        "DOCUMENTO": "",
        "TIENDA": tienda,
        "SECCION": "",
        "F. REGISTRO": fecha,
        "VALOR": valor,
        "IVA": iva,
        "RET. FUENTE": ret_fuente,
        "RET. IVA": ret_iva,
        "RET. ICA": ret_ica,
        "OTROS IMP.": otros_imp,
        "VALOR PAG": valor_pag,
        "DOC.SOPORTE": doc_soporte
    }

def parse_generic(line, voucher):
    """Intento de parseo genérico si no entra en los tipos anteriores."""
    for fn in (parse_DEC, parse_PMP, parse_FS, parse_CH):
        r = fn(line, voucher)
        if r:
            return r
    # Fallback
    mfecha = pat_fecha.search(line)
    if not mfecha:
        return None
    fecha = mfecha.group(0)
    descripcion = normalize_whitespace(line[len(voucher):mfecha.start()])
    nums = extract_numeric_values_after_date(line[mfecha.end():])
    while len(nums) < 8:
        nums.append("0")
    valor, iva, ret_fuente, ret_iva, ret_ica, otros_imp, valor_pag = nums[:7]
    doc_soporte = nums[7]
    return {
        "VOUCHER": voucher,
        "DESCRIPCION": descripcion,
        "DOCUMENTO": "",
        "TIENDA": "",
        "SECCION": "",
        "F. REGISTRO": fecha,
        "VALOR": valor,
        "IVA": iva,
        "RET. FUENTE": ret_fuente,
        "RET. IVA": ret_iva,
        "RET. ICA": ret_ica,
        "OTROS IMP.": otros_imp,
        "VALOR PAG": valor_pag,
        "DOC.SOPORTE": doc_soporte
    }

# -----------------------
# Dispatcher principal
# -----------------------
def parse_record(line: str):
    line = normalize_whitespace(line)
    m_start = re.match(rf"^{VOUCHERS}\b", line, flags=re.IGNORECASE)
    if not m_start:
        return None
    voucher = m_start.group(1).upper()

    if voucher in {"DCA", "DCF", "DND", "DAV", "DCC", "RPL"}:
        return parse_CH(line, voucher)  # sección fija
    if voucher == "DEC":
        return parse_DEC(line, voucher)
    if voucher in {"LTG", "FPM"}:
        return parse_PMP(line, voucher)
    if voucher == "FS":
        return parse_FS(line, voucher)
    if voucher == "CH":
        return parse_CH(line, voucher)

    return parse_generic(line, voucher)

# ------------------------
# Lectura y parseo PDF
# ------------------------
records = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        txt = page.extract_text()
        if not txt:
            continue
        lines = txt.split("\n")
        # Saltar encabezado
        for i, l in enumerate(lines):
            if l.strip().startswith("VOUCHER"):
                lines = lines[i+1:]
                break
        merged = merge_lines(lines)
        for rec_line in merged:
            parsed = parse_record(rec_line)
            if parsed:
                records.append(parsed)

# ------------------------
# Crear DataFrame
# ------------------------
cols = [
    "VOUCHER","DESCRIPCION","DOCUMENTO","TIENDA","SECCION",
    "F. REGISTRO","VALOR","IVA","RET. FUENTE","RET. IVA",
    "RET. ICA","OTROS IMP.","VALOR PAG","DOC.SOPORTE"
]
df = pd.DataFrame(records)
for c in cols:
    if c not in df.columns:
        df[c] = ""
df = df[cols]

# ------------------------
# Separar DOCUMENTO + TIENDA pegados (especial DEC)
# ------------------------
pat_merged = re.compile(r"^(\d{4}-\d{7,20})([A-ZÁÉÍÓÚÑ0-9 .-]+)$", flags=re.IGNORECASE)
for idx, row in df.iterrows():
    doc = str(row["DOCUMENTO"]).strip()
    tienda = str(row["TIENDA"]).strip()
    if not doc and tienda:
        m = pat_merged.match(tienda.replace(" ", ""))
        if m:
            df.at[idx, "DOCUMENTO"] = m.group(1)
            df.at[idx, "TIENDA"] = normalize_whitespace(m.group(2))

# ------------------------
# Extraer SECCION pegada en DESCRIPCION o TIENDA
# ------------------------
secciones_sorted = sorted(SECCIONES, key=lambda s: -len(s))

def extract_and_move_section_from_text(text: str):
    if not isinstance(text, str):
        return text, ""
    txt_low = text.lower()
    for sec in secciones_sorted:
        sec_low = sec.lower()
        idx = txt_low.rfind(sec_low)
        if idx != -1:
            seccion = text[idx:idx+len(sec)]
            new_text = re.sub(r"\s+", " ", (text[:idx] + text[idx+len(sec):]).strip())
            return new_text, seccion.upper()
    return text, ""

def fix_row_move_section(row):
    desc = str(row.get("DESCRIPCION", "")).strip()
    tienda = str(row.get("TIENDA", "")).strip()
    seccion_actual = str(row.get("SECCION", "")).strip()

    new_desc, found = extract_and_move_section_from_text(desc)
    if found:
        row["DESCRIPCION"] = new_desc
        row["SECCION"] = found
        tienda = tienda.replace(found, "").strip()
        row["TIENDA"] = re.sub(r"\s+", " ", tienda)
        return row

    new_tienda, found = extract_and_move_section_from_text(tienda)
    if found:
        row["TIENDA"] = new_tienda
        row["SECCION"] = found
        return row

    row["DESCRIPCION"] = re.sub(r"\s+", " ", desc)
    row["TIENDA"] = re.sub(r"\s+", " ", tienda)
    row["SECCION"] = seccion_actual
    return row

df = df.apply(fix_row_move_section, axis=1)

# Validar SECCION
df["SECCION"] = df["SECCION"].fillna("").str.upper().str.strip()
df.loc[~df["SECCION"].isin(SECCIONES), "SECCION"] = ""
df["TIENDA"] = df["TIENDA"].astype(str).str.strip()

# PARCHES
# Ajuste nombre de columnas:
df = df.rename(columns={
    "VALOR": "VALOR FAC.",
    "IVA": "IVA FAC."
})
# Parche para casos puntuales por no poder leer registros multi-línea
for idx, row in df.iterrows():
    # DESCRIPCION específica para RPL
    if row["VOUCHER"] == "RPL" and row["DESCRIPCION"].strip().upper() == "DESCUENTO":
        df.at[idx, "DESCRIPCION"] = "DESCUENTO COMERCIAL"

    # Ajuste de SECCION
    if row["SECCION"].strip().upper() == "DROGUE":
        df.at[idx, "SECCION"] = "DROGUER"

    # Ajustes específicos de TIENDA
    tienda_val = row["TIENDA"].strip().upper()
    if tienda_val == "PLAT - CROSS":
        df.at[idx, "TIENDA"] = "PLAT - CROSS DOCKIN"
    elif tienda_val == "PLAT - PLAT":
        df.at[idx, "TIENDA"] = "PLAT - PLAT BUCARAM"
    elif tienda_val == "PLAT - CROSSD":
        df.at[idx, "TIENDA"] = "PLAT - CROSSD AVERI"


In [113]:
# ------------------------
# Parche para casos puntuales por no poder leer registros multi-línea
# Mejora redacción de DESCRIPCION, SECCION y TIENDA
# ------------------------
for idx, row in df.iterrows():
    # DESCRIPCION específica para RPL
    if row["VOUCHER"] == "RPL" and row["DESCRIPCION"].strip().upper() == "DESCUENTO":
        df.at[idx, "DESCRIPCION"] = "DESCUENTO COMERCIAL"

    # Ajuste de SECCION
    if row["SECCION"].strip().upper() == "DROGUE":
        df.at[idx, "SECCION"] = "DROGUER"

    # Ajustes específicos de TIENDA
    tienda_val = row["TIENDA"].strip().upper()
    if tienda_val == "PLAT - CROSS":
        df.at[idx, "TIENDA"] = "PLAT - CROSS DOCKIN"
    elif tienda_val == "PLAT - PLAT":
        df.at[idx, "TIENDA"] = "PLAT - PLAT BUCARAM"
    elif tienda_val == "PLAT - CROSSD":
        df.at[idx, "TIENDA"] = "PLAT - CROSSD AVERI"


In [2]:
df.head(1000)

,VOUCHER,DESCRIPCION,DOCUMENTO,TIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,VALOR PAG,DOC.SOPORTE
0,FS,FACTURA VENTA,VPP1 2004529,ADM. FIDELIDAD - SE,,04/09/2025,48.364.550,8.874.450,0,0,0,5.291.000,57.239.000,0
1,FS,FACTURA VENTA,VPP1 2004621,ADM. FIDELIDAD - SE,,17/09/2025,5.429.700,996.300,0,0,0,594.000,6.426.000,0
2,FS,FACTURA VENTA,VPP1 2004622,ADM. FIDELIDAD - SE,,17/09/2025,35.192.500,6.457.500,0,0,0,3.850.000,41.650.000,0
3,FS,FACTURA VENTA,VPP2 2021716,ADM. JUMBO - SEDE,,01/11/2025,2.118.392.716,388.705.576,0,0,0,84.272.211,2.507.098.292,0
4,FS,FACTURA VENTA,VPP2 2022342,ADM. JUMBO - SEDE,,09/11/2025,909.145.377,166.819.814,0,0,0,36.166.897,1.075.965.191,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
869,DCC,DESCUENTO COMPRAS,,,PLATOS,08/11/2025,11.794,0,0,0,0,0,11.794,20.222.600.126.372
870,DAV,DESCUENTO APERTURA,,,PERFUME,08/11/2025,14.258.141,0,0,0,0,0,14.258.141,20.251.200.200.056
871,DCF,DSTO COMERCIAL FIJO,,,PERFUME,31/10/2025,7.026.140,0,0,0,0,0,7.026.140,20.181.200.033.536
872,DCF,DSTO COMERCIAL FIJO,,,PLATOS,31/10/2025,16.162.285,0,0,0,0,0,16.162.285,20.222.600.126.372
